# Title classification (local dev)

Experiment on real titles from Databricks. `llm_local.py` handles auth, batching, and costs;
you edit the prompt, model, and batch size below.

## 1. Load sample from Databricks

Edit the SQL below — table, filters, and `limit`. Uses `DATABRICKS_TOKEN` from `data/.env`.

In [ ]:
from optional.llm_classify.llm_local import query

SAMPLE_N = 25

# Edit table, filters, and columns. Tech-title filter avoids rows of only "Account Executive".
SQL = f"""
select posting_id, title
from team_c.analytics.stg_postings
where title is not null
  and (
    lower(title) rlike 'engineer|developer|software|programmer|architect'
    or lower(title) rlike 'data|analyst|scientist|ml|machine learning'
    or lower(title) rlike 'devops|sre|platform|infrastructure|cloud'
    or lower(title) rlike 'frontend|backend|full[ -]?stack|react|java|python'
  )
order by rand()
limit {SAMPLE_N}
"""

df = query(sql=SQL)
df

## 2. Models on the gateway

Run once to see ids you can pass to `chat()`. First call after idle can take ~60–90s while ACA
wakes up (default timeout 300s). Set `LITELLM_TIMEOUT_SECONDS` in the environment if needed.

In [ ]:
from optional.llm_classify.llm_local import list_models

list_models()

## 3. Classify

Edit `CATEGORIES`, `PROMPT` (keep `{numbered_items}`), `MODEL`, and `BATCH_SIZE`. Pass `categories=CATEGORIES` so labels match the dbt model.


In [ ]:
from optional.llm_classify.llm_local import classify

MODEL = "cheap"
BATCH_SIZE = 20

CATEGORIES = (
    "data_engineering",
    "data_science",
    "data_analytics",
    "ai_engineering",
    "ml_ai",
    "software_engineering",
    "backend",
    "frontend",
    "fullstack",
    "mobile",
    "devops",
    "security",
    "qa",
    "architecture",
    "network_engineering",
    "hardware",
    "product",
    "project_management",
    "management",
    "business_analysis",
    "design",
    "sales",
    "marketing",
    "operations",
    "support",
    "solutions_engineering",
    "finance",
    "hr",
    "recruiting",
    "legal",
    "technical_writing",
    "developer_relations",
    "blockchain",
    "other",
)

PROMPT = (
    "Classify each job title into exactly one job discipline.\n"
    f"Allowed disciplines: {', '.join(CATEGORIES)}.\n"
    "Choose the most specific discipline that matches the job title.\n"
    "Use other only when the title does not provide enough information "
    "or does not fit any allowed discipline.\n"
    'Reply with JSON only, like {"0": "backend", "1": "data_engineering"}.\n'
    "Use the numbers below as keys.\n\n"
    "{numbered_items}"
)

df["job_categories"] = classify(
    titles=df["title"],
    prompt=PROMPT,
    model=MODEL,
    batch_size=BATCH_SIZE,
    categories=CATEGORIES,
)
df
